In [1]:
# Setup and test cases 

%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")

from src.preprocessing import clean_label

examples = [
    "CB CARREFOUR MARKET SAINT-DENIS 28/10",
    "CARTE X6977 20/06 PLAYSTATION NETWORK",
    "PRLV SEPA DEEZER FR34ZZZ657871",
    "PRELEVEMENT SUEZ EAU REF FR50ZZZ983930",
    "FRAIS TENUE DE COMPTE 10/2026",
    "PAIEMENT CB 20/09 MONTREUIL ZETTLE *FOOD TRUCK",
    "CB NETFLIX.COM 12/03",
    "CB MICROSOFT*365 05/02",
    "CB G20 PARIS 15 03/04",
    "CB Café de la Gare 0412345 15/07",
]

for label in examples:
    print(f"{label:<48} -> {clean_label(label)}")

CB CARREFOUR MARKET SAINT-DENIS 28/10            -> CB CARREFOUR MARKET SAINT DENIS
CARTE X6977 20/06 PLAYSTATION NETWORK            -> CARTE PLAYSTATION NETWORK
PRLV SEPA DEEZER FR34ZZZ657871                   -> PRLV SEPA DEEZER
PRELEVEMENT SUEZ EAU REF FR50ZZZ983930           -> PRELEVEMENT SUEZ EAU REF
FRAIS TENUE DE COMPTE 10/2026                    -> FRAIS TENUE DE COMPTE
PAIEMENT CB 20/09 MONTREUIL ZETTLE *FOOD TRUCK   -> PAIEMENT CB MONTREUIL ZETTLE FOOD TRUCK
CB NETFLIX.COM 12/03                             -> CB NETFLIX COM
CB MICROSOFT*365 05/02                           -> CB MICROSOFT 365
CB G20 PARIS 15 03/04                            -> CB G20 PARIS 15
CB Café de la Gare 0412345 15/07                 -> CB CAFE DE LA GARE


In [2]:
# Apply the cleaning 

import pandas as pd

df = pd.read_csv("../data/raw/transactions.csv")
df["clean_label"] = df["label"].apply(clean_label)

df[["label", "clean_label", "category"]].sample(10, random_state=42)

,label,clean_label,category
3135,CB MEDIPOLE 21/10 CARTE X4721,CB MEDIPOLE CARTE,Health
2191,CB EBAY 08/02,CB EBAY,Shopping
1295,CB SAEMES 26/01,CB SAEMES,Transport
2902,PAIEMENT CB 01/05 TOULOUSE MUTUELLE MGEN,PAIEMENT CB TOULOUSE MUTUELLE MGEN,Health
168,PAIEMENT PAR CARTE NETTO 23/05,PAIEMENT PAR CARTE NETTO,Groceries
1128,PAIEMENT PAR CARTE ALLIANCE CARBURANTS 18/01,PAIEMENT PAR CARTE ALLIANCE CARBURANTS,Transport
4530,RETRAIT DAB 22/09 LILLE,RETRAIT DAB LILLE,Cash Withdrawal
4281,RETRAIT DAB 24/06 NICE,RETRAIT DAB NICE,Cash Withdrawal
4176,RETRAIT DAB 10/10 GRENOBLE,RETRAIT DAB GRENOBLE,Cash Withdrawal
1020,CB APRR 01/03,CB APRR,Transport


In [3]:
# How much variety is left after cleaning ? 

print(f"Rows: {len(df)}")
print(f"Distinct clean labels: {df['clean_label'].nunique()}")

conflicts = df.groupby("clean_label")["category"].nunique()
print(f"Clean labels assigned to more than one category: {(conflicts > 1).sum()}\n")

df.groupby("category")["clean_label"].nunique().sort_values()

Rows: 4960
Distinct clean labels: 2822
Clean labels assigned to more than one category: 0



category
Bank Fees               30
Transfers              167
Income                 174
Housing & Utilities    189
Cash Withdrawal        200
Subscriptions          222
Leisure & Travel       238
Transport              249
Health                 272
Groceries              343
Shopping               368
Restaurants            370
Name: clean_label, dtype: int64

In [4]:
# Save the cleaned dataset

columns = ["label", "clean_label", "category", "merchant", "payment_type"]
df[columns].to_csv("../data/processed/transactions_clean.csv", index=False)
print(f"Saved {len(df)} rows to data/processed/transactions_clean.csv")

Saved 4960 rows to data/processed/transactions_clean.csv


In [5]:
# Load and inspect the realistic set 

realistic = pd.read_csv("../data/test_realistic.csv")
realistic["clean_label"] = realistic["label"].apply(clean_label)

train = pd.read_csv("../data/processed/transactions_clean.csv")

unknown = set(realistic["category"]) - set(train["category"])
seen = realistic["clean_label"].isin(train["clean_label"])

print(f"{len(realistic)} realistic labels")
print(f"Unknown categories (typos): {unknown if unknown else 'none'}")
print(f"Clean labels already present in the synthetic data: {seen.sum()}\n")

realistic[["label", "clean_label", "category"]]

60 realistic labels
Unknown categories (typos): none
Clean labels already present in the synthetic data: 2



,label,clean_label,category
0,CARTE X4512 14/03 BIOCBON PARIS 11,CARTE BIOCBON PARIS 11,Groceries
1,CB INTERMARCHE SUPER 08/04/26,CB INTERMARCHE SUPER,Groceries
2,FACTURE CARTE DU 120326 AUCHAN SUPERMARCHE CAR...,FACTURE CARTE DU AUCHAN SUPERMARCHE CARTE 4974...,Groceries
3,PAIEMENT CB 0904 ROUEN MONOP,PAIEMENT CB ROUEN MONOP,Groceries
4,CB BOUCHERIE DU MARCHE 21/03,CB BOUCHERIE DU MARCHE,Groceries
5,CB BIG FERNAND 15/03,CB BIG FERNAND,Restaurants
6,FACTURE CARTE DU 180326 PRET A MANGER CARTE 49...,FACTURE CARTE DU PRET A MANGER CARTE 4974XXXXX...,Restaurants
7,CB SUMUP *LA TRATTORIA 22/03,CB SUMUP LA TRATTORIA,Restaurants
8,CB UBER *EATS HELP.UBER.COM 03/04,CB UBER EATS HELP UBER COM,Restaurants
9,PAIEMENT CB 2803 LYON BRASSERIE GEORGES,PAIEMENT CB LYON BRASSERIE GEORGES,Restaurants
